In [91]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from natsort import natsorted
from ultralytics import YOLO

from code_programm.path import get_path_weight_model

In [92]:
print(torch.cuda.device_count())
print(torch.cuda.get_device_name())
model = YOLO(get_path_weight_model('best.pt'))

1
NVIDIA GeForce GTX 1080 Ti


In [93]:
wheel_ets_train_x = pd.DataFrame(columns=[i for i in range(128 * 96)])

In [94]:
folder_path = Path('D:\Dataset_for_autopilot')
files_and_folders = os.listdir(folder_path)

# Фильтруем только папки
folders = [f for f in files_and_folders if os.path.isdir(os.path.join(folder_path, f))]

# Сортируем папки по дате изменения
sorted_folders = sorted(folders, key=lambda x: os.path.getmtime(os.path.join(folder_path, x)), reverse=True)

# Выводим список папок
print("Папки в папке {} отсортированы по дате изменения: ".format(folder_path))
for folder in sorted_folders:
    print(folder)

Папки в папке D:\Dataset_for_autopilot отсортированы по дате изменения: 
2024-04-02 19-12-38
2024-04-02 19-12-21
2024-04-02 19-11-55
2024-04-02 19-09-18
2024-04-02 19-07-31
2024-04-02 19-05-23
2024-04-02 19-01-56
2024-04-02 19-00-41
2024-04-02 18-58-46
2024-04-02 18-57-24
2024-04-02 18-55-25
new
2024-04-01 08-19-11
2024-04-01 08-18-41
2024-04-01 08-18-29
2024-04-01 08-18-18
2024-04-01 08-15-21
2024-04-01 08-13-16
2024-04-01 08-12-10
2024-04-01 08-10-13
2024-04-01 08-08-58
2024-04-01 08-07-13
2024-04-01 08-05-14
2024-04-01 08-03-03
2024-04-01 08-01-05
2024-04-01 07-58-48
2024-04-01 07-55-54
2024-04-01 07-51-15
2024-04-01 07-47-21
2024-04-01 07-42-36
2024-04-01 07-41-03
2024-04-01 07-36-51
2024-04-01 07-34-54
2024-04-01 07-32-35
2024-04-01 07-29-56
2024-04-01 07-26-37
num
2024-03-31 03-29-46
2024-03-31 03-28-05
2024-03-31 03-26-14
2024-03-31 03-24-22
2024-03-31 03-22-07
2024-03-31 03-17-11
2024-03-31 03-11-36
2024-03-31 03-05-57
numbers
Новая папка
2024-03-29 07-04-10
2024-03-29 07-01-32

In [95]:
paths = [r'D:\Dataset_for_autopilot\2024-04-02 19-12-38',
         r'D:\Dataset_for_autopilot\2024-04-02 19-12-21',
         r'D:\Dataset_for_autopilot\2024-04-02 19-11-55',
         r'D:\Dataset_for_autopilot\2024-04-02 19-09-18',
         r'D:\Dataset_for_autopilot\2024-04-02 19-07-31',
         r'D:\Dataset_for_autopilot\2024-04-02 19-05-23',
         r'D:\Dataset_for_autopilot\2024-04-02 19-01-56',
         r'D:\Dataset_for_autopilot\2024-04-02 19-00-41',
         r'D:\Dataset_for_autopilot\2024-04-02 18-58-46',
         r'D:\Dataset_for_autopilot\2024-04-02 18-57-24',
         r'D:\Dataset_for_autopilot\2024-04-02 18-55-25', ]


In [97]:
mass_results = []

for num_path in paths:
    path_i = os.path.join(num_path, f'road')
    if os.path.exists(f'{path_i}') and os.path.isdir(f'{path_i}'):
        png_files = [os.path.join(path_i, file) for file in os.listdir(path_i) if file.endswith('.png')]
        print("Полные пути к файлам в папке:")
        png_files = natsorted(png_files)
        print(png_files[0])
    else:
        print("Указанный путь не существует или не является папкой.")

    start_time = time.time()
    length = len(wheel_ets_train_x)

    combined_mask_old = torch.zeros((1, 1, 96, 128), device='cuda')
    combined_mask_new = torch.zeros((1, 1, 96, 128), device='cuda')

    for file in png_files:
        bgra_image = cv2.imread(file, cv2.IMREAD_UNCHANGED)
        bgr_image = cv2.cvtColor(bgra_image, cv2.COLOR_BGRA2BGR)
        results = model(bgr_image,
                        # imgsz=576,
                        conf=0.6,
                        # show=True,
                        device='cuda',
                        verbose=False)
        if results[0].masks is not None:
            combined_mask_old = combined_mask_new
            combined_mask_new.zero_()  # Reset the combined mask
            for i in results[0].masks.data:
                resized_mask = F.interpolate(i.unsqueeze(0).unsqueeze(0), size=(96, 128), mode='bilinear', align_corners=False)
                combined_mask_new += resized_mask.squeeze(0).unsqueeze(0)
        
        combined_tensor = combined_mask_old + combined_mask_new
        wheel_ets_train_x.loc[len(wheel_ets_train_x)] = combined_tensor.cpu().detach().numpy().flatten()

    print('Изображений:', len(wheel_ets_train_x) - length, '\nСек:', time.time() - start_time, '\nCек\изображение:',
          (time.time() - start_time) / (len(wheel_ets_train_x) - length), '\n')
print('Всего:', len(wheel_ets_train_x))
cv2.destroyAllWindows()

Полные пути к файлам в папке:
D:\Dataset_for_autopilot\2024-04-02 19-12-38\road\2024-04-02 19-12-38_0.png
Изображений: 80 
Сек: 2.148219585418701 
Cек\изображение: 0.026852744817733764 

Полные пути к файлам в папке:
D:\Dataset_for_autopilot\2024-04-02 19-12-21\road\2024-04-02 19-12-21_0.png
Изображений: 256 
Сек: 7.813781499862671 
Cек\изображение: 0.030522583983838558 

Полные пути к файлам в папке:
D:\Dataset_for_autopilot\2024-04-02 19-11-55\road\2024-04-02 19-11-55_0.png
Изображений: 231 
Сек: 5.401519775390625 
Cек\изображение: 0.023383202490868508 

Полные пути к файлам в папке:
D:\Dataset_for_autopilot\2024-04-02 19-09-18\road\2024-04-02 19-09-18_0.png
Изображений: 591 
Сек: 16.541505336761475 
Cек\изображение: 0.027989010722100634 

Полные пути к файлам в папке:
D:\Dataset_for_autopilot\2024-04-02 19-07-31\road\2024-04-02 19-07-31_0.png
Изображений: 1817 
Сек: 73.95315146446228 
Cек\изображение: 0.040700688753143796 

Полные пути к файлам в папке:
D:\Dataset_for_autopilot\2024

In [98]:
wheel_ets_train_x.to_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_road.csv', index=False)

In [100]:
wheel_ets_train_x_ = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_road.csv')
len(wheel_ets_train_x_)

13118

In [89]:
combined_mask_old = torch.zeros((1, 1, 96, 128), device='cuda')
combined_mask_new = torch.zeros((1, 1, 96, 128), device='cuda')

bgra_image = cv2.imread(png_files[0], cv2.IMREAD_UNCHANGED)
bgr_image = cv2.cvtColor(bgra_image, cv2.COLOR_BGRA2BGR)
results = model(bgr_image,
                        # imgsz=576,
                        conf=0.6,
                        # show=True,
                        device='cuda',
                        verbose=False)
if results[0].masks is not None:
    combined_mask_old = combined_mask_new
    combined_mask_new.zero_()  # Reset the combined mask
    for i in results[0].masks.data:
        resized_mask = F.interpolate(i.unsqueeze(0).unsqueeze(0), size=(96, 128), mode='bilinear', align_corners=False)
        combined_mask_new += resized_mask.squeeze(0).unsqueeze(0)

combined_tensor = combined_mask_old + combined_mask_new
print(type(combined_tensor.flatten().detach()[10223]))
print(combined_tensor.flatten().detach()[9223])
print(type(combined_tensor.cpu().detach().numpy().flatten()))
print(combined_tensor.cpu().detach().numpy().flatten()[9223])

cv2.waitKey(0)
cv2.destroyAllWindows()

<class 'torch.Tensor'>
tensor(2., device='cuda:0')
<class 'numpy.ndarray'>
2.0
